In [ ]:
pip install lightgbm scikit-learn

In [ ]:
FEATURES = ['zero_ratio_12', 'demand_rate', 'q75_nonzero_demand', 'mean_nonzero_demand',
            'std_nonzero_demand', 'median_nonzero_demand', 'croston_alpha_demand',
            'croston_alpha_interval', 'croston_forecast', 'mean_demand_interval',
            'rolling_mean_12', 'rolling_std_12', 'ema_slow', 'ema_fast', 'macd',
            'sales_ratio', 'rolling_mean_4', 'rolling_std_4', 'recent_trend_4',
            'sbc_category', 'sin_1', 'sin_2', 'cos_1', 'adi', 'cv2']

BLOCK_DAYS = 28  # 4 hafta

FINAL_DECISION = {
    'Erratic':'model',
    'Lumpy': 'model',
    'LowData': 'baseline',
    'Intermittent': 'baseline',
    'Smooth': 'model'
}

In [ ]:
def teknorot(pred, actual, under_penalty=2.0, over_penalty=1.0):
    pred = np.asarray(pred)
    actual = np.asarray(actual)
    under = np.maximum(actual - pred, 0)   # eksik tahmin tutarı
    over  = np.maximum(pred - actual, 0)   # fazla tahmin tutarı
    return (under_penalty * under.sum() + over_penalty * over.sum()) / actual.sum()

def bias(pred, actual):
    pred = np.asarray(pred)
    actual = np.asarray(actual)
    return (pred - actual).sum() / actual.sum()   # negatif = eksik tahmin eğilimi

def teknorot_per_unit(pred, actual, under_penalty=2.0, over_penalty=1.0):
    under = np.maximum(actual - pred, 0)
    over  = np.maximum(pred - actual, 0)
    return under_penalty * under + over_penalty * over

def rmsse(actual, pred, train_actual):
    """
    actual, pred: real and predict values of val and test periods
    train_actual: weekly real sales series of this product in train set
    """
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    train_actual = np.asarray(np.diff(train_actual), dtype=float)

    numerator = np.mean((actual - pred) ** 2)
    denominator = np.mean(np.diff(train_actual) ** 2)

    if denominator == 0:
        return np.nan

    return np.sqrt(numerator / denominator)

def wape(pred, actual):
    return np.abs(pred - actual).sum() / actual.sum()

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from google.colab import userdata

data_path = userdata.get('DATA_PATH')
df = pd.read_csv(data_path)

df['week'] = pd.to_datetime(df['week'])

In [ ]:
WARMUP_WEEKS = 12

df2 = df[df['history_weeks'] >= WARMUP_WEEKS].copy()

In [ ]:
folds = [
    (pd.Timestamp("2024-11-01"), pd.Timestamp("2025-01-01")),
    (pd.Timestamp("2025-01-01"), pd.Timestamp("2025-03-01")),
    (pd.Timestamp("2025-03-01"), pd.Timestamp("2025-05-01")),
]

In [ ]:
results = []

for fold_i, (train_end, val_end) in enumerate(folds, start=1):
  train = df2[df2['week'] < train_end].copy()
  val   = df2[(df2['week'] >= train_end) & (df2['week'] < val_end)].copy()

  if len(train) == 0 or len(val) == 0:
        print(f"Fold {fold_i}: yetersiz veri, atlanıyor.")
        continue

  train['sbc_category'] = train['sbc_category'].astype('category')
  cat_categories = train['sbc_category'].cat.categories
  val['sbc_category'] = pd.Categorical(val['sbc_category'], categories=cat_categories)

  X_train, y_train = train[FEATURES], train['quantity']
  X_val, y_val = val[FEATURES], val['quantity']

  model = lgb.LGBMRegressor(
        objective="tweedie",
        tweedie_variance_power=1.5,
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        verbose=-1,
    )
  model.fit(
        X_train, y_train,
        eval_metric='tweedie',
        eval_set=[(X_val, y_val)],
        categorical_feature=['sbc_category'],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
    )

  preds = np.maximum(model.predict(X_val), 0)

  mean_52 = (train.sort_values('week').groupby('product')['quantity'].apply(lambda x: x.tail(52).mean()))
  baseline_pred = val['product'].map(mean_52).fillna(0).values

  pred_df = val[['product', 'week', 'sbc_category', 'quantity']].copy()
  pred_df['actual'] = y_val.values
  pred_df['pred'] = preds
  pred_df['baseline'] = baseline_pred

  pred_df['block'] = ((pred_df['week'] - pred_df['week'].min()).dt.days // BLOCK_DAYS)

  agg = pred_df.groupby(['product', 'block']).agg(
        pred=('pred', 'sum'),
        baseline=('baseline', 'sum'),
        actual=('actual', 'sum'),
        sbc_category=('sbc_category', 'first'),
    )

  agg['router'] = agg.apply(
        lambda row: row['pred'] if FINAL_DECISION.get(row['sbc_category'], 'baseline') == 'model' else row['baseline'],
        axis=1
    )

  agg['blend5050'] = 0.5 * agg['pred'] + 0.5 * agg['baseline']

  methods = ['pred', 'baseline', 'router', 'blend5050']
  method_labels = {'pred': 'Model', 'baseline': 'Baseline', 'router': 'Router', 'blend5050': '50/50 Blend'}

  fold_comparison = {}
  for m in methods:
      fold_comparison[f'wape_{m}'] = wape(agg[m], agg['actual'])
      fold_comparison[f'teknorot_{m}'] = teknorot(agg[m], agg['actual'])

  print(f"\n  === Fold {fold_i}: Yöntem Karşılaştırması ===")
  for m in methods:
      print(f"    {method_labels[m]:<12} WAPE={fold_comparison[f'wape_{m}']:.3f}  Teknorot={fold_comparison[f'teknorot_{m}']:.3f}")

  teknorot_model    = teknorot(agg['pred'], agg['actual'])
  teknorot_baseline = teknorot(agg['baseline'], agg['actual'])
  bias_model    = bias(agg['pred'], agg['actual'])
  bias_baseline = bias(agg['baseline'], agg['actual'])

  model_teknorot_err    = teknorot_per_unit(agg['pred'], agg['actual'])
  baseline_teknorot_err = teknorot_per_unit(agg['baseline'], agg['actual'])
  teknorot_win_rate = (model_teknorot_err < baseline_teknorot_err).mean()

  wape_model = np.abs(agg['pred'] - agg['actual']).sum() / agg['actual'].sum()
  wape_baseline = np.abs(agg['baseline'] - agg['actual']).sum() / agg['actual'].sum()

  model_err = np.abs(agg['pred'] - agg['actual'])
  baseline_err = np.abs(agg['baseline'] - agg['actual'])
  win_rate = (model_err < baseline_err).mean()

  seg_rows = []
  for cat in agg['sbc_category'].dropna().unique():
      mask = agg['sbc_category'] == cat
      if mask.sum() > 0:
          seg_win = (model_err[mask] < baseline_err[mask]).mean()
          seg_rows.append((str(cat), int(mask.sum()), seg_win))

  for category in agg['sbc_category'].unique():
      mask = agg['sbc_category'] == category
      cat_win_rate = (model_teknorot_err[mask] < baseline_teknorot_err[mask]).mean()
      print(f"{category}: n={mask.sum()}, teknorot_win_rate={cat_win_rate:.1%}")

  results.append({
        'fold': fold_i,
        'train_end': train_end.date(),
        'val_end': val_end.date(),
        'n_val_rows': len(val),
        'wape_model_block': wape_model,
        'wape_baseline_block': wape_baseline,
        'win_rate_overall': win_rate,
        **fold_comparison
    })

  rmsse_rows = []

  for product in val['product'].unique():
      val_mask = val['product'] == product
      train_mask = train['product'] == product
      pred_mask = pred_df['product'] == product

      actual_vals = pred_df.loc[pred_mask, 'quantity'].values
      pred_vals   = pred_df.loc[pred_mask, 'pred'].values          # model tahmini
      baseline_vals = pred_df.loc[pred_mask, 'baseline'].values    # mean52
      train_vals  = train.loc[train_mask, 'quantity'].values
      category    = pred_df.loc[pred_mask, 'sbc_category'].iloc[0]

      rmsse_model    = rmsse(actual_vals, pred_vals, train_vals)
      rmsse_baseline = rmsse(actual_vals, baseline_vals, train_vals)

      rmsse_rows.append({
          'fold': fold_i,
          'product': product,
          'sbc_category': category,
          'rmsse_model': rmsse_model,
          'rmsse_baseline': rmsse_baseline,
      })

  rmsse_df = pd.DataFrame(rmsse_rows)

  print(f"\n=== Fold {fold_i} ({train_end.date()} -> {val_end.date()}) ===")
  print(f"  n_val_rows: {len(val)}")
  print(f"  Blok WAPE  - model: {wape_model:.3f} | baseline(mean52): {wape_baseline:.3f}")
  print(f"  Win rate (genel): {win_rate:.1%}")
  print(f"  Teknorot -   model: {teknorot_model:.3f} | baseline: {teknorot_baseline:.3f}")
  print(f"  Bias     -   model: {bias_model:+.3f} | baseline: {bias_baseline:+.3f}")
  print(f"  Teknorot  Win Rate:{teknorot_win_rate:.1%}")
  for cat, n, wr in seg_rows:
      print(f"    {cat:<13} n={n:<5} win_rate={wr:.1%}")


  === Fold 1: Yöntem Karşılaştırması ===
    Model        WAPE=0.948  Teknorot=1.460
    Baseline     WAPE=0.969  Teknorot=1.513
    Router       WAPE=0.930  Teknorot=1.459
    50/50 Blend  WAPE=0.954  Teknorot=1.479
Lumpy: n=1358, teknorot_win_rate=56.8%
Intermittent: n=2368, teknorot_win_rate=38.9%
LowData: n=5076, teknorot_win_rate=7.1%
Erratic: n=849, teknorot_win_rate=52.3%
Smooth: n=96, teknorot_win_rate=58.3%


/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



=== Fold 1 (2024-11-01 -> 2025-01-01) ===
  n_val_rows: 29030
  Blok WAPE  - model: 0.948 | baseline(mean52): 0.969
  Win rate (genel): 26.0%
  Teknorot -   model: 1.460 | baseline: 1.513
  Bias     -   model: -0.075 | baseline: -0.120
  Teknorot  Win Rate:26.2%
    Lumpy         n=1358  win_rate=56.3%
    Intermittent  n=2368  win_rate=38.8%
    LowData       n=5076  win_rate=7.1%
    Erratic       n=849   win_rate=51.8%
    Smooth        n=96    win_rate=58.3%

  === Fold 2: Yöntem Karşılaştırması ===
    Model        WAPE=0.880  Teknorot=1.436
    Baseline     WAPE=0.892  Teknorot=1.447
    Router       WAPE=0.874  Teknorot=1.442
    50/50 Blend  WAPE=0.879  Teknorot=1.430
Intermittent: n=1673, teknorot_win_rate=43.3%
LowData: n=3541, teknorot_win_rate=7.1%
Lumpy: n=944, teknorot_win_rate=53.7%
Erratic: n=554, teknorot_win_rate=56.0%
Smooth: n=74, teknorot_win_rate=47.3%


/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



=== Fold 2 (2025-01-01 -> 2025-03-01) ===
  n_val_rows: 26929
  Blok WAPE  - model: 0.880 | baseline(mean52): 0.892
  Win rate (genel): 27.0%
  Teknorot -   model: 1.436 | baseline: 1.447
  Bias     -   model: -0.232 | baseline: -0.217
  Teknorot  Win Rate:27.0%
    Intermittent  n=1673  win_rate=43.2%
    LowData       n=3541  win_rate=7.1%
    Lumpy         n=944   win_rate=53.9%
    Erratic       n=554   win_rate=56.7%
    Smooth        n=74    win_rate=43.2%

  === Fold 3: Yöntem Karşılaştırması ===
    Model        WAPE=1.044  Teknorot=1.548
    Baseline     WAPE=1.095  Teknorot=1.565
    Router       WAPE=1.029  Teknorot=1.536
    50/50 Blend  WAPE=1.063  Teknorot=1.547
Lumpy: n=1492, teknorot_win_rate=62.3%
LowData: n=5641, teknorot_win_rate=7.6%
Intermittent: n=2609, teknorot_win_rate=45.8%
Erratic: n=859, teknorot_win_rate=58.3%
Smooth: n=124, teknorot_win_rate=56.5%


/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



=== Fold 3 (2025-03-01 -> 2025-05-01) ===
  n_val_rows: 31866
  Blok WAPE  - model: 1.044 | baseline(mean52): 1.095
  Win rate (genel): 29.1%
  Teknorot -   model: 1.548 | baseline: 1.565
  Bias     -   model: +0.037 | baseline: +0.156
  Teknorot  Win Rate:29.1%
    Lumpy         n=1492  win_rate=62.3%
    LowData       n=5641  win_rate=7.6%
    Intermittent  n=2609  win_rate=45.8%
    Erratic       n=859   win_rate=58.4%
    Smooth        n=124   win_rate=56.5%


In [ ]:
summary = pd.DataFrame(results)
print("\n=== FOLD ÖZETİ ===")
#print(summary.to_string(index=False))
print(f"\nOrtalama model WAPE:    {summary['wape_model_block'].mean():.3f}  (std: {summary['wape_model_block'].std():.3f})")
print(f"Ortalama baseline WAPE: {summary['wape_baseline_block'].mean():.3f}  (std: {summary['wape_baseline_block'].std():.3f})")
print(f"Ortalama win_rate:      {summary['win_rate_overall'].mean():.1%}  (std: {summary['win_rate_overall'].std():.1%})")



=== FOLD ÖZETİ ===

Ortalama model WAPE:    0.957  (std: 0.082)
Ortalama baseline WAPE: 0.985  (std: 0.103)
Ortalama win_rate:      27.4%  (std: 1.6%)


In [ ]:
rmsse_df.to_csv('rmsse.csv', encoding='utf-8', index=False)